In [1]:
import os
import sys
import pandas as pd
import joblib

In [2]:
from sklearn.model_selection import (
    train_test_split
)

from sklearn.ensemble import (
    RandomForestRegressor
)

from sklearn.model_selection import (
    RandomizedSearchCV,
    cross_val_score
)

from sklearn.preprocessing import (
    StandardScaler
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


In [3]:
# Get project root
project_root = os.path.abspath("..")

# Add to Python path
sys.path.append(project_root)

print(project_root)

/home/chaithanaya/Documents/ai-route-optimizer


In [4]:

# =====================================================
# PATHS
# =====================================================

feature_path = os.path.join(
    project_root,
    "data",
    "feature_dataset.csv"
)

model_dir = os.path.join(
    project_root,
    "models"
)

os.makedirs(
    model_dir,
    exist_ok=True
)

model_path = os.path.join(
    model_dir,
    "route_eta_model.pkl"
)

columns_path = os.path.join(
    model_dir,
    "model_columns.pkl"
)


In [5]:

# =====================================================
# LOAD DATASET
# =====================================================

df = pd.read_csv(
    feature_path
)

print("Dataset Shape:")
print(df.shape)


Dataset Shape:
(480, 21)


In [6]:
print("First 5 rows")
print(df.head())

First 5 rows
   trip_id driver_id day_of_week  month  quarter  hour  weekend_flag  \
0        1        D1   Wednesday      4        2     9             0   
1        2        D2   Wednesday      4        2     9             0   
2        3        D3   Wednesday      4        2     9             0   
3        4        D4   Wednesday      4        2     8             0   
4        5        D5   Wednesday      4        2    10             0   

   weekly_pattern  total_distance_km  historical_travel_time  ...  \
0              14             258.71                  348.95  ...   
1              14              36.97                   87.45  ...   
2              14             204.33                  227.37  ...   
3              14             299.28                  409.30  ...   
4              14              77.56                  133.47  ...   

   optimized_duration_min  optimized_distance_km  average_speed  daily_visits  \
0                  281.05                 157.19          

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 480 entries, 0 to 479
Data columns (total 21 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   trip_id                 480 non-null    int64  
 1   driver_id               480 non-null    object 
 2   day_of_week             480 non-null    object 
 3   month                   480 non-null    int64  
 4   quarter                 480 non-null    int64  
 5   hour                    480 non-null    int64  
 6   weekend_flag            480 non-null    int64  
 7   weekly_pattern          480 non-null    int64  
 8   total_distance_km       480 non-null    float64
 9   historical_travel_time  480 non-null    float64
 10  stop_count              480 non-null    int64  
 11  optimized_duration_min  480 non-null    float64
 12  optimized_distance_km   480 non-null    float64
 13  average_speed           480 non-null    float64
 14  daily_visits            480 non-null    in

In [8]:
# =====================================================
# DROP NULLS
# =====================================================

df = df.dropna()


In [9]:
# =====================================================
# TARGET VARIABLE
# =====================================================

TARGET = "optimized_duration_min"


In [10]:
# =====================================================
# REMOVE NON-FEATURE COLUMNS
# =====================================================
drop_columns = [

    "trip_id",

    "driver_id",

    "day_of_week",

    "total_distance_km",
    "historical_travel_time",
"duration_per_stop",
"average_speed",
"past_route_efficiency",

    TARGET
]

X = df.drop(
    columns=drop_columns
)

y = df[TARGET]


In [11]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 480 entries, 0 to 479
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   month                  480 non-null    int64  
 1   quarter                480 non-null    int64  
 2   hour                   480 non-null    int64  
 3   weekend_flag           480 non-null    int64  
 4   weekly_pattern         480 non-null    int64  
 5   stop_count             480 non-null    int64  
 6   optimized_distance_km  480 non-null    float64
 7   daily_visits           480 non-null    int64  
 8   area_cluster           480 non-null    int64  
 9   avg_latitude           480 non-null    float64
 10  avg_longitude          480 non-null    float64
 11  distance_per_stop      480 non-null    float64
dtypes: float64(4), int64(8)
memory usage: 45.1 KB


In [12]:
# =====================================================
# ONE-HOT ENCODING
# =====================================================

X = pd.get_dummies(
    X,
    drop_first=True
)


In [13]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 480 entries, 0 to 479
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   month                  480 non-null    int64  
 1   quarter                480 non-null    int64  
 2   hour                   480 non-null    int64  
 3   weekend_flag           480 non-null    int64  
 4   weekly_pattern         480 non-null    int64  
 5   stop_count             480 non-null    int64  
 6   optimized_distance_km  480 non-null    float64
 7   daily_visits           480 non-null    int64  
 8   area_cluster           480 non-null    int64  
 9   avg_latitude           480 non-null    float64
 10  avg_longitude          480 non-null    float64
 11  distance_per_stop      480 non-null    float64
dtypes: float64(4), int64(8)
memory usage: 45.1 KB


In [14]:

# =====================================================
# SAVE TRAINING COLUMNS
# =====================================================

joblib.dump(
    X.columns.tolist(),
    columns_path
)


['/home/chaithanaya/Documents/ai-route-optimizer/models/model_columns.pkl']

In [15]:

# =====================================================
# TRAIN TEST SPLIT
# =====================================================

X_train, X_test, y_train, y_test = (
    train_test_split(

        X,
        y,

        test_size=0.2,

        random_state=42
    )
)

print("\nTraining Shape:")
print(X_train.shape)

print("\nTesting Shape:")
print(X_test.shape)



Training Shape:
(384, 12)

Testing Shape:
(96, 12)


In [16]:
# =====================================================
# STANDARD SCALER
# =====================================================

scaler = StandardScaler()

X_train = scaler.fit_transform(
    X_train
)

X_test = scaler.transform(
    X_test
)


In [17]:
scaler_path = os.path.join(
    model_dir,
    "scaler.pkl"
)

In [18]:
joblib.dump(
    scaler,
    scaler_path
)

['/home/chaithanaya/Documents/ai-route-optimizer/models/scaler.pkl']

In [19]:

# =====================================================
# MODEL
# =====================================================

model = RandomForestRegressor(

    n_estimators=200,

    max_depth=12,


    n_jobs=-1
)

In [20]:
# =====================================================
# TRAIN MODEL
# =====================================================

print("\nTraining Random Forest...")

model.fit(
    X_train,
    y_train
)

print("\nTraining Complete!")


Training Random Forest...

Training Complete!


In [21]:
# =====================================================
# PREDICTIONS
# =====================================================

predictions = model.predict(
    X_test
)


In [22]:
# =====================================================
# EVALUATION
# =====================================================

mae = mean_absolute_error(
    y_test,
    predictions
)

mse = mean_squared_error(
    y_test,
    predictions
)

rmse = mse ** 0.5

r2 = r2_score(
    y_test,
    predictions
)

# =====================================================
# PRINT METRICS
# =====================================================

print("\nModel Evaluation")

print("-" * 40)

print(f"MAE  : {mae:.2f}")

print(f"MSE  : {mse:.2f}")

print(f"RMSE : {rmse:.2f}")

print(f"R2   : {r2:.4f}")


Model Evaluation
----------------------------------------
MAE  : 7.58
MSE  : 169.53
RMSE : 13.02
R2   : 1.0000


In [23]:
# =====================================================
# FEATURE IMPORTANCE
# =====================================================

importance_df = pd.DataFrame({

    "feature": X.columns,

    "importance": model.feature_importances_
})

importance_df = importance_df.sort_values(
    by="importance",
    ascending=False
)

print("\nTop Important Features:\n")

print(
    importance_df.head(10)
)



Top Important Features:

                  feature    importance
6   optimized_distance_km  9.999426e-01
7            daily_visits  1.710121e-05
5              stop_count  1.494368e-05
10          avg_longitude  9.059373e-06
9            avg_latitude  6.224764e-06
11      distance_per_stop  6.196431e-06
4          weekly_pattern  1.629007e-06
8            area_cluster  1.536966e-06
0                   month  3.704435e-07
3            weekend_flag  2.339078e-07


In [24]:
param_grid = {

    "n_estimators": [
        100,
        200,
        300
    ],

    "max_depth": [
        8,
        10,
        12,
        15,
        None
    ],

    "min_samples_split": [
        2,
        5,
        10
    ],

    "min_samples_leaf": [
        1,
        2,
        4
    ]
}

In [25]:
base_model = RandomForestRegressor(
    random_state=42,
    n_jobs=-1
)

In [26]:
random_search = RandomizedSearchCV(

    estimator=base_model,

    param_distributions=param_grid,

    n_iter=15,

    cv=5,

    verbose=2,

    random_state=42,

    n_jobs=-1,

    scoring="r2"
)

In [27]:
print("\nRunning Hyperparameter Tuning...")

random_search.fit(
    X_train,
    y_train
)


Running Hyperparameter Tuning...
Fitting 5 folds for each of 15 candidates, totalling 75 fits
[CV] END max_depth=15, min_samples_leaf=2, min_samples_split=10, n_estimators=300; total time=   1.8s
[CV] END max_depth=15, min_samples_leaf=2, min_samples_split=10, n_estimators=300; total time=   1.3s
[CV] END max_depth=15, min_samples_leaf=2, min_samples_split=10, n_estimators=300; total time=   1.5s
[CV] END max_depth=15, min_samples_leaf=2, min_samples_split=10, n_estimators=300; total time=   1.6s
[CV] END max_depth=15, min_samples_leaf=2, min_samples_split=10, n_estimators=300; total time=   1.2s
[CV] END max_depth=12, min_samples_leaf=2, min_samples_split=5, n_estimators=200; total time=   1.0s
[CV] END max_depth=12, min_samples_leaf=2, min_samples_split=5, n_estimators=200; total time=   0.8s
[CV] END max_depth=12, min_samples_leaf=2, min_samples_split=5, n_estimators=200; total time=   0.9s
[CV] END max_depth=12, min_samples_leaf=2, min_samples_split=5, n_estimators=200; total time

/home/chaithanaya/anaconda3/lib/python3.11/site-packages/numpy/ma/core.py:2820: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


RandomizedSearchCV(cv=5,
                   estimator=RandomForestRegressor(n_jobs=-1, random_state=42),
                   n_iter=15, n_jobs=-1,
                   param_distributions={'max_depth': [8, 10, 12, 15, None],
                                        'min_samples_leaf': [1, 2, 4],
                                        'min_samples_split': [2, 5, 10],
                                        'n_estimators': [100, 200, 300]},
                   random_state=42, scoring='r2', verbose=2)

In [28]:
model = random_search.best_estimator_

In [29]:
cv_scores = cross_val_score(

    model,

    X_train,

    y_train,

    cv=5,

    scoring="r2"
)

print("\nCross Validation Scores:")

print(cv_scores)

print(

    "\nAverage CV Score:",

    round(cv_scores.mean(), 4)
)


Cross Validation Scores:
[0.99265072 0.99999509 0.98452255 0.99999028 0.99999127]

Average CV Score: 0.9954


In [30]:
print("\nBest Parameters:")

print(
    random_search.best_params_
)


Best Parameters:
{'n_estimators': 300, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_depth': None}


In [31]:

# =====================================================
# MODEL
# =====================================================

model_rf = RandomForestRegressor(

    n_estimators=300,
    min_samples_split=2,
    min_samples_leaf=1,
    max_depth=None,


    n_jobs=-1
)

In [32]:
model_rf.fit(
    X_train,
    y_train
)

RandomForestRegressor(n_estimators=300, n_jobs=-1)

In [33]:
# =====================================================
# PREDICTIONS
# =====================================================

y_pred = model_rf.predict(
    X_test
)


In [34]:
# =====================================================
# EVALUATION
# =====================================================

mae = mean_absolute_error(
    y_test,
    y_pred
)

mse = mean_squared_error(
    y_test,
    y_pred
)

rmse = mse ** 0.5

r2 = r2_score(
    y_test,
    y_pred
)

adr2 = r2_score(
    y_test,
    y_pred
)

# =====================================================
# PRINT METRICS
# =====================================================

print("\nModel Evaluation")

print("-" * 40)

print(f"MAE  : {mae:.2f}")

print(f"MSE  : {mse:.2f}")

print(f"RMSE : {rmse:.2f}")

print(f"R2   : {r2:.4f}")


Model Evaluation
----------------------------------------
MAE  : 7.44
MSE  : 161.29
RMSE : 12.70
R2   : 1.0000


In [35]:
# =====================================================
# SAVE MODEL
# =====================================================

joblib.dump(
    model,
    model_path
)

print("\nModel Saved!")

print(f"\nPath: {model_path}")


Model Saved!

Path: /home/chaithanaya/Documents/ai-route-optimizer/models/route_eta_model.pkl
